# AgentRegistry, end to end: scaffold a dice agent, add approved MCP tools, deploy to kagent **and** AWS Bedrock AgentCore, then govern the tools

A developer builds an agent with `arctl`, runs it locally, pulls in **approved MCP tool servers** from the registry catalog, publishes the agent, and deploys the *same* published agent to two runtimes (Solo Enterprise for kagent on kind, and AWS Bedrock AgentCore). Finally a platform owner restricts which MCP tools the agent may call with an **AccessPolicy**, enforced at an agentgateway waypoint.

> **Kernel:** pick **Bash** (top-right). One-time engineer setup lives in `scripts/` (see `README.md`).

## What you're running

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 980 620" width="100%" font-family="-apple-system,Segoe UI,Helvetica,Arial,sans-serif">
  <style>
    .ns{rx:10;ry:10;stroke-width:1.5}
    .pill{fill:#ffffff;stroke:#cbd5e1;stroke-width:1;rx:7;ry:7}
    .t{font-size:13px;fill:#0f172a}
    .nt{font-size:13px;font-weight:700}
    .sub{font-size:11px;fill:#475569}
    .lbl{font-size:11px;fill:#334155}
    .ar{stroke:#64748b;stroke-width:1.6;fill:none;marker-end:url(#a)}
  </style>
  <defs><marker id="a" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#64748b"/></marker></defs>

  <!-- kind cluster -->
  <rect x="20" y="58" width="630" height="544" rx="14" fill="#f8fafc" stroke="#1e293b" stroke-width="2"/>
  <text x="38" y="82" class="nt" fill="#1e293b">kind cluster · agentcore-demo</text>
  <text x="38" y="99" class="sub">ambient mesh (Solo Istio ztunnel + agentgateway waypoint)</text>

  <!-- kagent ns -->
  <rect x="38" y="112" width="594" height="150" class="ns" fill="#e0f2fe" stroke="#0369a1"/>
  <text x="52" y="132" class="nt" fill="#0369a1">namespace: kagent — Solo Enterprise for kagent</text>
  <rect x="52" y="144" width="150" height="42" class="pill"/><text x="62" y="162" class="t">kagent controller</text><text x="62" y="177" class="sub">+ OBO, OIDC</text>
  <rect x="214" y="144" width="120" height="42" class="pill"/><text x="224" y="162" class="t">kagent UI</text><text x="224" y="177" class="sub">:18007</text>
  <rect x="346" y="144" width="130" height="42" class="pill"/><text x="356" y="162" class="t">agentdemo</text><text x="356" y="177" class="sub">ADK agent pod</text>
  <rect x="488" y="144" width="132" height="42" class="pill"/><text x="498" y="162" class="t">agentgateway</text><text x="498" y="177" class="sub">waypoint (enforce)</text>
  <rect x="52" y="198" width="190" height="42" class="pill"/><text x="62" y="216" class="t">MCP: everything-server</text><text x="62" y="231" class="sub">sum/echo/printenv…</text>
  <rect x="254" y="198" width="150" height="42" class="pill"/><text x="264" y="216" class="t">MCP: my-mcp</text><text x="264" y="231" class="sub">word_count</text>
  <rect x="416" y="198" width="204" height="42" class="pill"/><text x="426" y="216" class="t">AccessPolicy (deny printenv)</text><text x="426" y="231" class="sub">kagent CR, enforced at waypoint</text>

  <!-- solo-enterprise ns -->
  <rect x="38" y="276" width="594" height="118" class="ns" fill="#fef3c7" stroke="#92400e"/>
  <text x="52" y="296" class="nt" fill="#92400e">namespace: solo-enterprise — management + observability</text>
  <rect x="52" y="308" width="150" height="42" class="pill"/><text x="62" y="326" class="t">Enterprise UI</text><text x="62" y="341" class="sub">:18090 · Tracing</text>
  <rect x="214" y="308" width="170" height="42" class="pill"/><text x="224" y="326" class="t">OTel collector</text><text x="224" y="341" class="sub">OTLP :4317</text>
  <rect x="396" y="308" width="150" height="42" class="pill"/><text x="406" y="326" class="t">ClickHouse</text><text x="406" y="341" class="sub">otel_traces_json</text>
  <text x="52" y="370" class="sub">agent spans ▸ collector ▸ ClickHouse ▸ Enterprise UI Tracing tab</text>

  <!-- keycloak ns -->
  <rect x="38" y="408" width="594" height="64" class="ns" fill="#dcfce7" stroke="#166534"/>
  <text x="52" y="430" class="nt" fill="#166534">namespace: keycloak — single sign-on</text>
  <text x="52" y="450" class="sub">realm “solo” · OIDC issuer keycloak.localtest.me:18080 · alice / alice (field-fte → Admin)</text>
  <text x="52" y="465" class="sub">one login for arctl, the kagent controller, kagent UI and the Enterprise UI</text>

  <!-- local registry -->
  <rect x="38" y="486" width="594" height="104" class="ns" fill="#f1f5f9" stroke="#475569"/>
  <text x="52" y="506" class="nt" fill="#334155">in-cluster: Solo Istio · agentgateway · local image registry (:5001)</text>
  <text x="52" y="526" class="sub">Gateway API + GatewayClasses; ztunnel enrols the kagent namespace into the ambient mesh</text>
  <text x="52" y="546" class="sub">the waypoint is where MCP tool calls are authorized against the AccessPolicy</text>
  <text x="52" y="566" class="sub">Anthropic Claude is the model for the agent on both runtimes</text>

  <!-- arctl daemon (control plane) -->
  <rect x="690" y="112" width="268" height="120" rx="12" fill="#1e293b"/>
  <text x="706" y="136" class="nt" fill="#f8fafc">arctl daemon (Docker)</text>
  <text x="706" y="156" class="sub" fill="#cbd5e1">AgentRegistry control plane</text>
  <text x="706" y="174" class="sub" fill="#cbd5e1">UI/API :12121 · catalog of</text>
  <text x="706" y="190" class="sub" fill="#cbd5e1">approved MCP servers + skills</text>
  <text x="706" y="212" class="sub" fill="#cbd5e1">connected runtimes: kagent, AgentCore</text>

  <!-- Anthropic -->
  <rect x="690" y="270" width="268" height="64" rx="12" fill="#f5f5f4" stroke="#78350f"/>
  <text x="706" y="294" class="nt" fill="#78350f">Anthropic Claude</text>
  <text x="706" y="314" class="sub">model · claude-haiku-4-5</text>

  <!-- AWS AgentCore -->
  <rect x="690" y="372" width="268" height="118" rx="12" fill="#fff7ed" stroke="#9a3412"/>
  <text x="706" y="396" class="nt" fill="#9a3412">AWS Bedrock AgentCore</text>
  <text x="706" y="416" class="sub">runtime #2 (external)</text>
  <text x="706" y="434" class="sub">clones agent source from git,</text>
  <text x="706" y="450" class="sub">runs the same published agent</text>
  <text x="706" y="472" class="sub">connected once at setup (CF role + ECR)</text>

  <!-- arrows -->
  <path class="ar" d="M690,176 L636,176"/><text x="640" y="170" class="lbl" text-anchor="end">deploy → kagent (alice OIDC)</text>
  <path class="ar" d="M824,232 L824,270"/><text x="832" y="256" class="lbl">model</text>
  <path class="ar" d="M824,334 C824,360 700,360 500,372" transform="translate(0,0)"/>
  <path class="ar" d="M758,232 C758,330 690,420 700,420"/><text x="700" y="356" class="lbl" text-anchor="end">deploy #2 → AgentCore</text>
</svg>

> The developer drives everything through **arctl** → the **AgentRegistry** control plane, which deploys the *same* published agent to two runtimes. One **Keycloak** login spans the whole platform; agent **traces** flow to **ClickHouse** and render in the **Enterprise UI**; MCP tool calls are governed by an **AccessPolicy** enforced at an **agentgateway waypoint**.

## Open the consoles

Run this once **in a terminal** — it port-forwards and opens the kagent UI, the Enterprise UI (Tracing), and the AgentRegistry UI:

```bash
./scripts/open-consoles.sh
```

| Console | URL | Login | What it's for |
|---|---|---|---|
| AgentRegistry UI | http://localhost:12121 | — | catalog, runtimes, deployed instances |
| kagent UI | http://localhost:18007 | alice / alice | chat with the agent, tool calls |
| **Enterprise UI** | **http://localhost:18090** | alice / alice | **Dashboard, Agents, Tracing, Access Policies** |
| AWS Bedrock AgentCore | the AWS console (manual) | — | the second runtime |

> **Tracing lives in the Enterprise UI** (`:18090` → **Tracing**), not the kagent UI.

## Connect to the platform

Loads creds, puts `arctl` on the path, mints a catalog token.

In [ ]:
rm -rf agentdemo

In [ ]:
[ -d agentregistry-agentcore-kind ] && cd agentregistry-agentcore-kind || :; source scripts/connect.sh

## 1. Browse the registry: approved tools, skills, and runtimes

Rather than wiring arbitrary MCP servers, developers pull from an **approved catalog**. Open the **AgentRegistry UI** (http://localhost:12121) → **Tool Servers**, or list it from the CLI. While we're here, list the **skills** (reusable instruction packs) and the **runtimes** the registry can deploy to — `kind-kagent` (Solo Enterprise for kagent) and `aws-agentcore` (AWS Bedrock AgentCore), both connected at setup.

In [ ]:
arctl get mcpservers
echo
arctl get skills
echo
arctl get runtimes

## 2. Create the agent, wired to the approved tools

Scaffold the agent and reference the approved servers straight from the catalog with `--mcp` (repeatable). arctl records them in `agent.yaml` under `spec.mcpServers`, resolved into tool servers beside the agent at deploy time. The generated agent also has two **local** tools, `roll_die` and `check_prime`.

In [ ]:
arctl init agent agentdemo --framework adk --language python --model-provider anthropic --model-name claude-haiku-4-5 --mcp demo/everything-server@latest --mcp demo/my-mcp@latest

In [ ]:
cat agentdemo/agentdemo/agent.py

In [ ]:
cat agentdemo/agent.yaml

## 3. Build and run it locally

Build the image:

In [ ]:
arctl build ./agentdemo

Then chat with the agent **in a terminal** — `arctl run` is interactive, so copy the command below and run it in your shell:

```bash
arctl run ./agentdemo
```

Ask it (paste into the chat):

```text
Roll a 20-sided die and tell me whether the result is prime.
```

It uses the local `roll_die` + `check_prime` tools. The approved catalog tools (`sum`, `echo`, `word_count`, ...) come online when the agent is **deployed**: the registry stands up the MCP servers beside it (you'll see them next).

## 4. Build (multi-arch), publish, and push the source

Multi-arch so the same image runs on kagent (arm64 here) and AgentCore (amd64).

In [ ]:
arctl build ./agentdemo --platform linux/amd64,linux/arm64 --push

In [ ]:
arctl apply -f agentdemo/agent.yaml

AgentCore clones the agent **source** from git at deploy time, so push it to your agent repo (the engineer set `AGENT_GIT_URL` in `.env.local`):

In [ ]:
./scripts/git-push.sh

## 5. Deploy onto kagent (runtime #1) — with arctl

On a Kagent runtime the agent's MCP tools don't appear on their own — you deploy the **approved MCP tool servers** to the runtime first, then deploy the agent, which wires itself to them. Both are an `arctl apply` of an AgentRegistry **Deployment** bound to `kind-kagent`.

First, deploy the two approved MCP servers. The registry stands each up as a kagent MCP server (named `demo-<server>`), already fronted by an **agentgateway waypoint** (the enforcement point we use in §8):

In [ ]:
# Deploy the approved MCP tool servers onto kind-kagent (each its own Deployment).
arctl apply -f yaml/deploy-mcp.yaml
echo "waiting for the MCP servers to be Ready..."
kubectl --context kind-agentcore-demo -n kagent rollout status deploy/demo-everything-server --timeout=180s
kubectl --context kind-agentcore-demo -n kagent rollout status deploy/demo-my-mcp --timeout=180s
kubectl --context kind-agentcore-demo -n kagent get mcpserver

In [ ]:
# Now the agent. The registry binds it to kind-kagent AND auto-derives the agent's
# MCP_SERVERS_CONFIG from the MCP servers we just deployed (it skips any that aren't
# Ready — that's why we deployed them first). envsubst fills the model key.
cat yaml/deploy-kagent.yaml
echo "---- deploy it with arctl ----"
envsubst < yaml/deploy-kagent.yaml | arctl apply -f -

In [ ]:
kubectl --context kind-agentcore-demo -n kagent get pods

## 6. Deploy the same agent to AWS Bedrock AgentCore (runtime #2)

The *same* published agent, now to a second runtime. The AWS Bedrock AgentCore platform was **connected once at setup** (cross-account role + registered runtime), so this just makes the agent multi-cloud, pushes its image and source, and deploys it against the `aws-agentcore` platform — then waits for it to go READY:

In [ ]:
source scripts/aws-login.sh

In [ ]:
./scripts/agentcore-deploy.sh

In [ ]:
./scripts/ac-invoke.sh "Roll a 13-sided die, add 5 to the result, then tell me if it is prime."

## 7. Run it, then watch the trace in the Enterprise UI

Chat with `agentdemo` in the **kagent UI** (http://localhost:18007) — paste the prompt below. It exercises the whole chain (a local tool, an MCP tool, then another local tool):

```text
Roll a 13-sided die, add 5 to the result, then tell me if it is prime.
```

Now open the **Enterprise UI** (http://localhost:18090) → **Tracing**. The run shows up as a span tree — `invocation → call_llm → generate_content (claude-haiku) → execute_tool roll_die → sum → check_prime` — with the model and token usage on each LLM span. (The kagent UI shows the tool calls inline in the chat; the full OTEL trace lives in the Enterprise UI.)

## 8. Govern the MCP tools with an AccessPolicy (in the UI)

The `everything-server` exposes a sensitive `printEnv` tool. A platform owner restricts the agent to least privilege — **in the UI**.

First, see what the agent can call today: in the **kagent UI** (http://localhost:18007) open a chat with `agentdemo` and ask —

```text
What tools do you have?
```

You'll see the full set, including `printEnv`.

Now create the policy in the **Enterprise UI** (http://localhost:18090) → **Access Policies** → **Create New Access Policy**:

- **Subjects:** Agent → `agentdemo`
- **Target:** Target Type **MCP Server** → `demo-everything-server`
- **Action:** **ALLOW**, tools: **`sum`**

This is an allowlist — on the everything-server the agent may now call **only** `sum`; `printEnv`, `echo` and the rest are denied. (Prefer the CLI? `./scripts/accesspolicy-on.sh` applies the identical policy.)

The MCP server is **already** behind an agentgateway waypoint (the registry labels it `kagent.solo.io/waypoint=true` at deploy time) — that waypoint is the enforcement point. This makes it explicit / idempotent:

In [ ]:
# The everything-server is already waypoint-labelled by the registry; confirm/ensure it.
kubectl --context kind-agentcore-demo -n kagent label mcpserver demo-everything-server kagent.solo.io/waypoint=true --overwrite
kubectl --context kind-agentcore-demo -n kagent rollout status deploy/mcpserver-demo-everything-server-waypoint --timeout=120s 2>/dev/null || true
echo "✓ demo-everything-server is behind the agentgateway waypoint — the AccessPolicy is enforced there"

In [ ]:
# The policy you created in the UI is a declarative kagent CR — show it:
kubectl --context kind-agentcore-demo -n kagent get accesspolicy
echo
kubectl --context kind-agentcore-demo -n kagent get accesspolicy -o yaml

Back in the **kagent UI**, start a **New Chat** with `agentdemo` and ask again — the everything-server now offers only `sum`; `printEnv` and the rest are gone:

> A new chat re-lists the tools through the waypoint — no restart needed. (If the list looks unchanged, the agent is still on its startup tool cache: `kubectl --context kind-agentcore-demo -n kagent rollout restart deploy/agentdemo`.)

```text
What tools do you have?
```

## Reset / teardown

Run any of these **in a terminal** (copy each line):

```bash
./scripts/accesspolicy-off.sh   # restore the full tool list
./scripts/reset.sh              # back to start (clears agentdemo, deployments)
./scripts/cleanup.sh            # full teardown (cluster, daemon, registry, AWS)
```